In [2]:
from datetime import datetime, timezone
from pathlib import Path
import requests

from download_single_file import OpenDataAPI
from download_full_dataset import (
    download_files_date_range,
    download_full_dataset,
    create_resilient_session,
)

# Configuration
API_TOKEN_NORMAL = "eyJvcmciOiI1ZTU1NGUxOTI3NGE5NjAwMDEyYTNlYjEiLCJpZCI6ImY4ZjZhMmYxYTE3YTQ5ZTM4NThlMGIxZWYyODRjYzY3IiwiaCI6Im11cm11cjEyOCJ9"
API_TOKEN_FULL_DATASET = "eyJvcmciOiI1ZTU1NGUxOTI3NGE5NjAwMDEyYTNlYjEiLCJpZCI6ImMxNzA3OTMxOWVhNzRjNmM5NTJkMTg0YzA5ZGFhN2UyIiwiaCI6Im11cm11cjEyOCJ9"
API_TOKEN_HOURLY_DATASET_FULL = "eyJvcmciOiI1ZTU1NGUxOTI3NGE5NjAwMDEyYTNlYjEiLCJpZCI6IjVlZWM3ZmEyMWE2ZDQ4N2VhYzRiNGY4MWRiZTgzZTU3IiwiaCI6Im11cm11cjEyOCJ9"
DATASET_NAME = "hourly-in-situ-meteorological-observations-validated"
DATASET_VERSION = "1.0"
BASE_URL = "https://api.dataplatform.knmi.nl/open-data/v1"
BASE_TARGET = r"C:\Users\floris\Desktop\MSC\thesis_msc\data\hourly_data_validated"

# Concurrency / rate-limit settings.
# KNMI allows 100 req/s; leave headroom at 90. Each download costs one API call
# (the /url endpoint) plus one non-API S3 fetch, so ~90 files/s is the ceiling.
MAX_WORKERS = 64
RATE_LIMIT_PER_SEC = 95

# Create resilient session with automatic retries and timeouts.
# pool_size must be >= MAX_WORKERS to avoid connection-pool contention.
print("Creating resilient session with retry strategy...")
session = create_resilient_session(
    token=API_TOKEN_HOURLY_DATASET_FULL,
    pool_size=max(128, MAX_WORKERS * 2),
)
print("Session created. Connection errors will be retried automatically with exponential backoff.")

Creating resilient session with retry strategy...
Session created. Connection errors will be retried automatically with exponential backoff.


In [2]:
# Chunk 2: Download the files listed in missing_10min.json one-by-one
# via OpenDataAPI.get_file_url + download_file_from_temporary_download_url.

import json
from download_single_file import (
    OpenDataAPI,
    download_file_from_temporary_download_url,
)

MISSING_JSON = Path(r"C:\Users\floris\Desktop\MSC\thesis_msc\missing_10min.json")

with open(MISSING_JSON, "r") as f:
    missing_files = json.load(f)["missing"]

print(f"Found {len(missing_files)} missing files to download.")

api = OpenDataAPI(api_token=API_TOKEN_FULL_DATASET)
base_target = Path(BASE_TARGET)

downloaded, failed = [], []
for i, filename in enumerate(missing_files, 1):
    # filename format: KMDS__OPER_P___10M_OBS_L2_YYYYMMDDHHMM.nc
    stamp = filename.split("_")[-1].replace(".nc", "")
    year, month = stamp[:4], stamp[4:6]
    target_dir = base_target / year / month
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / filename

    if target_path.exists():
        continue

    try:
        resp = api.get_file_url(DATASET_NAME, DATASET_VERSION, filename)
        url = resp.get("temporaryDownloadUrl")
        if not url:
            print(f"[{i}/{len(missing_files)}] No URL for {filename}: {resp}")
            failed.append(filename)
            continue
        download_file_from_temporary_download_url(url, str(target_path))
        downloaded.append(filename)
        if i % 50 == 0:
            print(f"[{i}/{len(missing_files)}] downloaded so far: {len(downloaded)}")
    except Exception as e:
        print(f"[{i}/{len(missing_files)}] Failed {filename}: {e}")
        failed.append(filename)

print(f"\nDone. Downloaded: {len(downloaded)}, Failed: {len(failed)}")
if failed:
    print("First failures:", failed[:10])

Found 961 missing files to download.
[1/961] No URL for KMDS__OPER_P___10M_OBS_L2_201204051750.nc: {'message': 'Not Found'}
[2/961] No URL for KMDS__OPER_P___10M_OBS_L2_201204051800.nc: {'message': 'Not Found'}
[3/961] No URL for KMDS__OPER_P___10M_OBS_L2_201204051810.nc: {'message': 'Not Found'}
[4/961] No URL for KMDS__OPER_P___10M_OBS_L2_201204051820.nc: {'message': 'Not Found'}
[5/961] No URL for KMDS__OPER_P___10M_OBS_L2_201204051830.nc: {'message': 'Not Found'}
[6/961] No URL for KMDS__OPER_P___10M_OBS_L2_201204051840.nc: {'message': 'Not Found'}
[7/961] No URL for KMDS__OPER_P___10M_OBS_L2_201307021150.nc: {'message': 'Not Found'}
[8/961] No URL for KMDS__OPER_P___10M_OBS_L2_201307021200.nc: {'message': 'Not Found'}
[9/961] No URL for KMDS__OPER_P___10M_OBS_L2_201307021210.nc: {'message': 'Not Found'}
[10/961] No URL for KMDS__OPER_P___10M_OBS_L2_201307021220.nc: {'message': 'Not Found'}
[11/961] No URL for KMDS__OPER_P___10M_OBS_L2_201507241750.nc: {'message': 'Not Found'}
[12/

In [3]:
# Chunk 3: Download full hourly dataset under the 100 req/s KNMI API limit.
# Each file = 1 KNMI API call (/url) + 1 S3 fetch (not rate-limited).
# RateLimiter inside download_full_dataset caps the aggregate API rate to
# RATE_LIMIT_PER_SEC across all workers.

print("Building session with the full-dataset hourly token...")
session_full = create_resilient_session(
    token=API_TOKEN_HOURLY_DATASET_FULL,
    pool_size=max(128, MAX_WORKERS * 2),
)

print(f"Starting full dataset download (max_workers={MAX_WORKERS}, " f"rate_limit={RATE_LIMIT_PER_SEC} req/s)...")

downloaded_full = download_full_dataset(
    session=session_full,
    base_url=BASE_URL,
    dataset_name=DATASET_NAME,
    dataset_version=DATASET_VERSION,
    base_target=BASE_TARGET,
    max_keys=500,
    overwrite=False,  # set to false to not re-download files we already have
    max_workers=MAX_WORKERS,
    rate_limit_per_sec=RATE_LIMIT_PER_SEC,
)

print(f"\nFull dataset download complete!")
print(f"Total files downloaded: {len(downloaded_full)}")

INFO:download_full_dataset:Retrieve dataset files with query params: {'maxKeys': '500'}


Building session with the full-dataset hourly token...
Starting full dataset download (max_workers=64, rate_limit=95 req/s)...


INFO:download_full_dataset:Retrieve dataset files with query params: {'maxKeys': '500', 'nextPageToken': 'eyJmaWxlX25hbWUiOiAiaG91cmx5LW9ic2VydmF0aW9ucy12YWxpZGF0ZWQtMTk1MTAxMjEtMTkubmMiLCAiaWQiOiAiaG91cmx5LWluLXNpdHUtbWV0ZW9yb2xvZ2ljYWwtb2JzZXJ2YXRpb25zLXZhbGlkYXRlZF8xLjBfaG91cmx5LW9ic2VydmF0aW9ucy12YWxpZGF0ZWQtMTk1MTAxMjEtMTkubmMifQ=='}
INFO:download_full_dataset:Retrieve dataset files with query params: {'maxKeys': '500', 'nextPageToken': 'eyJmaWxlX25hbWUiOiAiaG91cmx5LW9ic2VydmF0aW9ucy12YWxpZGF0ZWQtMTk1MTAyMTEtMTUubmMiLCAiaWQiOiAiaG91cmx5LWluLXNpdHUtbWV0ZW9yb2xvZ2ljYWwtb2JzZXJ2YXRpb25zLXZhbGlkYXRlZF8xLjBfaG91cmx5LW9ic2VydmF0aW9ucy12YWxpZGF0ZWQtMTk1MTAyMTEtMTUubmMifQ=='}
INFO:download_full_dataset:Retrieve dataset files with query params: {'maxKeys': '500', 'nextPageToken': 'eyJmaWxlX25hbWUiOiAiaG91cmx5LW9ic2VydmF0aW9ucy12YWxpZGF0ZWQtMTk1MTAzMDQtMTEubmMiLCAiaWQiOiAiaG91cmx5LWluLXNpdHUtbWV0ZW9yb2xvZ2ljYWwtb2JzZXJ2YXRpb25zLXZhbGlkYXRlZF8xLjBfaG91cmx5LW9ic2VydmF0aW9ucy12YWxpZGF0ZWQtMTk1


Full dataset download complete!
Total files downloaded: 660456
